# 🧪 AI-Scientist v1 on Google Colab

## Fully Automated Scientific Discovery — Powered by DeepSeek V4 Flash 🚀

This notebook runs [SakanaAI's AI-Scientist v1](https://github.com/SakanaAI/AI-Scientist) — a fully automated scientific discovery pipeline.

### Pipeline overview:
1. 💡 **Idea Generation** — LLM generates novel research ideas
2. 🧪 **Experimentation** — Automated template-based experiments
3. 📝 **Write-up** — Auto-generates LaTeX paper
4. 🔍 **Review** — LLM reviews its own paper

> ⚠️ **Warning:** This pipeline executes LLM-generated code. Review outputs carefully. Use in a sandboxed environment.

> 💰 **Cost:** ~$0.50–3 per run with DeepSeek V4 Flash (10× cheaper than GPT-4o)

## 🔑 Setup Instructions

You need API keys for at least one LLM provider. Supported models:

| Provider | Env Variable | Model Name | Cost |
|----------|-------------|------------|------|
| **DeepSeek** | `DEEPSEEK_API_KEY` | `deepseek-v4-flash` | ~$0.50–1/run 💥 |
| **DeepSeek** | `DEEPSEEK_API_KEY` | `deepseek-chat` | ~$0.50–1/run |
| **OpenAI** | `OPENAI_API_KEY` | `gpt-4o-mini` | ~$1–2/run |
| **Anthropic** | `ANTHROPIC_API_KEY` | `claude-3-5-sonnet-20241022` | ~$15/run |

### How to add secrets in Colab:
1. Click the **🔑 Key** icon in the left sidebar
2. Click **+ Add new secret**
3. Add your API key(s)

DeepSeek V4 Flash (`deepseek-v4-flash`) is the most **cost-effective** option — get your key at:
👉 [https://platform.deepseek.com/api_keys](https://platform.deepseek.com/api_keys)

> If `deepseek-v4-flash` is not available, use `deepseek-chat` instead.

Secrets are loaded using `from google.colab import userdata`.

In [ ]:
import os, sys, json
from pathlib import Path
from google.colab import userdata

WORKING_DIR = Path("/content")
PROJECT_DIR = WORKING_DIR / "AI-Scientist"
os.chdir(WORKING_DIR)

# Clone repo
!git clone https://github.com/SakanaAI/AI-Scientist.git
%cd AI-Scientist

# Install LaTeX (smaller package for Colab)
!apt-get update -qq && apt-get install -y -qq texlive-latex-base texlive-latex-extra texlive-fonts-recommended texlive-bibtex-extra poppler-utils > /dev/null 2>&1

# Install Python packages
!pip install -q -r requirements.txt

# Verify GPU
import torch
if torch.cuda.is_available():
    print(f"\N{white heavy check mark} GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\N{warning sign} No GPU detected")

In [ ]:
# Load API keys from Colab secrets
# Add them in: Runtime → Secrets → Add new secret
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or ""
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY") or ""
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY") or ""
os.environ["S2_API_KEY"] = userdata.get("S2_API_KEY") or ""

# Show which keys are set
for k in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "DEEPSEEK_API_KEY", "S2_API_KEY"]:
    status = "\N{white heavy check mark} Set" if os.environ.get(k) else "\N{cross mark} Not set"
    print(f"   {status}: {k}")

In [ ]:
%cd {PROJECT_DIR}

print("\N{open mail box with raised flag} Preparing NanoGPT data...")
!python data/enwik8/prepare.py
!python data/shakespeare_char/prepare.py
!python data/text8/prepare.py

print("\N{person running} Running baseline experiment (this takes 10-30 minutes)...")
%cd templates/nanoGPT
!python experiment.py --out_dir run_0
!python plot.py
%cd {PROJECT_DIR}
print("\N{white heavy check mark} NanoGPT baseline ready!")

## 🚀 How to Launch AI Scientist

The main launch command:

```bash
python launch_scientist.py --model "deepseek-v4-flash" --experiment nanoGPT_lite --num-ideas 2
```

### Available templates:
| Template | Description | Colab-friendly |
|----------|-------------|:------:|
| `nanoGPT_lite` | Small GPT language model 🔤 | ✅ Yes |
| `nanoGPT` | Full GPT language model | ⚠️ May be heavy |
| `2d_diffusion` | 2D diffusion denoising 🎨 | ⚠️ May be heavy |
| `grokking` | Algorithmic grokking 🧮 | ⚠️ May be heavy |
| `probes` | Probing classifiers | ⚠️ Unknown |

> 💡 **Recommendation:** Start with `nanoGPT_lite` — it's designed for low-resource environments.

### Recommended models:
| Model | Quality | Cost |
|-------|---------|------|
| `deepseek-v4-flash` | Good | ~$0.50–1/run 💥 |
| `deepseek-chat` | Good | ~$0.50–1/run |
| `gpt-4o-mini` | Good | ~$1–2/run |
| `claude-3-5-sonnet-20241022` | Best | ~$15/run |

> For 2D Diffusion: first install NPEET (see Cell 9 tips)

In [ ]:
%cd {PROJECT_DIR}

# ✏️ CONFIGURE YOUR RUN HERE
MODEL = "deepseek-v4-flash"      # LLM model (fallback: "deepseek-chat")
EXPERIMENT = "nanoGPT_lite"      # Template: nanoGPT_lite, 2d_diffusion, grokking, nanoGPT
NUM_IDEAS = 2                    # Number of ideas to try

print("=" * 50)
print("\N{rocket} Launching AI Scientist v1")
print("=" * 50)
print(f"   Model:      {MODEL}")
print(f"   Experiment: {EXPERIMENT}")
print(f"   Ideas:      {NUM_IDEAS}")
print()

!python launch_scientist.py \
    --model {MODEL} \
    --experiment {EXPERIMENT} \
    --num-ideas {NUM_IDEAS} \
    2>&1

print()
print("\N{white heavy check mark} AI Scientist run complete!")

In [ ]:
%cd {PROJECT_DIR}

print("\N{open file folder} Results:\n")

# Find PDFs
import glob
pdfs = sorted(glob.glob("**/*.pdf", recursive=True))
if pdfs:
    for pdf in pdfs:
        size = os.path.getsize(pdf) / 1e6
        print(f"   \N{page facing up} {pdf} ({size:.1f} MB)")
else:
    print("   No PDFs found yet.")

# Find experiment folders
folders = sorted(glob.glob("templates/**/run_*", recursive=True))
if folders:
    print()
    for f in folders:
        print(f"   \N{file folder} {f}")

print("\n\N{light bulb} Download: Files sidebar (\N{open file folder} icon) \u2192 right-click \u2192 Download")

## 💡 Tips

### ⏰ Session timeouts
- Colab session disconnects after **~12h of inactivity**
- Save intermediate results to Google Drive

### 💾 Save to Google Drive
```python
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/AI-Scientist /content/drive/MyDrive/
```

### 🎨 2D Diffusion template
If using `2d_diffusion_lite`, first run:
```python
!git clone https://github.com/gregversteeg/NPEET.git
%cd NPEET
!pip install .
%cd {PROJECT_DIR}
```

### 🐘 CUDA Out of Memory
- Use **Lite** experiment variants (lower memory footprint)
- Restart runtime: Runtime → Restart and run all
- Reduce `NUM_IDEAS` to 1

### 📋 Logs
- Check `log.txt` in experiment output folders for detailed logs
- Use Runtime → View runtime logs for Colab-level diagnostics